In [2]:
import shutil

In [4]:
# shutil.rmtree('/kaggle/working/Yoruba-to-english-translation')

!git clone https://github.com/Jmkernes/PAR-Transformer-XL.git
!git clone https://github.com/Frontman-11/Yoruba-to-english-translation.git

Cloning into 'PAR-Transformer-XL'...
remote: Enumerating objects: 263, done.
remote: Counting objects: 100% (263/263), done.
remote: Compressing objects: 100% (160/160), done.
remote: Total 263 (delta 130), reused 211 (delta 86), pack-reused 0 (from 0)
Receiving objects: 100% (263/263), 11.26 MiB | 19.16 MiB/s, done.
Resolving deltas: 100% (130/130), done.
Cloning into 'Yoruba-to-english-translation'...
remote: Enumerating objects: 28, done.
remote: Counting objects: 100% (28/28), done.
remote: Compressing objects: 100% (27/27), done.
remote: Total 28 (delta 15), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (28/28), 7.16 KiB | 7.16 MiB/s, done.
Resolving deltas: 100% (15/15), done.


In [5]:
import sys

PATHS = ["/kaggle/working/PAR-Transformer-XL", "/kaggle/working/Yoruba-to-english-translation"]
for PATH in PATHS:
    if PATH in sys.path:
        sys.path.remove(PATH)  
    sys.path.append(PATH)

import importlib
# import transformer_components
# importlib.reload(transformer_components)

from transformer_components import PositionalEncoding, DecoderTransformerBlock, EncoderTransformerBlock
from adaptive_softmax import AdaptiveSoftmax
import tensorflow as tf

In [13]:
N = 3
n_units = 128    #First Dense layer unit in each feedforward block
num_heads = 8
proj_dims = []
proj_factor = 4
embed_size = 256
dropout_rate = 0.1
vocab_size = 50_000
MAX_SEQ_LENGTH = 512
cutoffs = [5_000, 20_000, 50_000]


encoder_embed_layer =  tf.keras.layers.Embedding(
    input_dim=vocab_size,
    output_dim=embed_size,
    embeddings_initializer='glorot_uniform',
    mask_zero=True,
)

decoder_embed_layer =  tf.keras.layers.Embedding(
    input_dim=vocab_size,
    output_dim=embed_size,
    embeddings_initializer='glorot_uniform',
    mask_zero=True,
)

encoder_block = EncoderTransformerBlock(embed_size,
                                  n_units=n_units,
                                  num_heads=num_heads,
                                  dropout_rate=dropout_rate,
                                  N=N,
                                )
decoder_block = DecoderTransformerBlock(embed_size,
                                  n_units=n_units,
                                  num_heads=num_heads,
                                  dropout_rate=dropout_rate,
                                  N=N,
                                )
positional_encoding = PositionalEncoding(max_seq_length=MAX_SEQ_LENGTH, embed_size=embed_size)

adaptive_softmax = AdaptiveSoftmax(cutoffs=cutoffs, proj_factor=proj_factor, proj_dims=proj_dims)

In [ ]:
encoder_input_ids = None
decoder_input_ids = None
encoder_pad_mask = tf.math.not_equal(encoder_input_ids, 0)[:, tf.newaxis]
decoder_pad_mask = tf.math.not_equal(decoder_input_ids, 0)[:, tf.newaxis]
causal_mask = tf.linalg.band_part(tf.ones((batch_max_len_dec, batch_max_len_dec), tf.bool), -1, 0)
attention_mask1 = causal_mask & decoder_pad_mask


encoder_input = positional_encoding(encoder_embed_layer)
encoder_output = encoder_block(inputs=encoder_input, attention_mask=encoder_pad_mask)

decoder_input = positional_encoding(decoder_embed_layer)
decoder_output = encoder_block(inputs=decoder_input, encoder_output=encoder_output, attention_mask1=attention_mask1, attention_mask2=encoder_pad_mask)

y_proba = adaptive_softmax(inp=decoder_output)

0
